In [ ]:
### K-Means Clustering for Customer Segmentation ###

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

# Preparing data for clustering
X_clustering = df_feat.drop(columns=['churn'])

# [Scaling numerical features and one-hot encodes categorical features and getting feature names]


# Determining Optimal Number of Clusters (K) using Elbow Method
inertia = []
max_k = 10 # Test up to 10 clusters
for k in range(1, max_k + 1):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    kmeans.fit(X_clustering_processed)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, max_k + 1), inertia, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(range(1, max_k + 1))
plt.grid(True)
plt.show()


# Determining Optimal K using Silhouette Score
silhouette_scores = []
for k in range(2, max_k + 1):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    kmeans.fit(X_clustering_processed)
    score = silhouette_score(X_clustering_processed, kmeans.labels_)
    silhouette_scores.append(score)

plt.figure(figsize=(10, 6))
plt.plot(range(2, max_k + 1), silhouette_scores, marker='o')
plt.title('Silhouette Score for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(range(2, max_k + 1))
plt.grid(True)
plt.show()


# Applying K-Means Clustering with a Chosen K
chosen_k = 4  # chosen based on above plots
kmeans = KMeans(n_clusters=chosen_k, random_state=RANDOM_SEED, n_init=10)

# [Adding cluster labels to the original dataframe]

print(f"Customers grouped into {chosen_k} clusters. Cluster distribution:")
print(df_clustered['cluster'].value_counts().sort_index().to_markdown(numalign="left", stralign="left"))


# Profile Clusters
cluster_profile = df_clustered.groupby('cluster').agg({
    'credit_score': 'mean',
    'age': 'mean',
    'tenure': 'mean',
    'balance': 'mean',
    'products_number': 'mean',
    'credit_card': lambda x: x.mode()[0], # Mode for binary/categorical
    'active_member': lambda x: x.mode()[0], # Mode for binary/categorical
    'estimated_salary': 'mean',
    'churn': 'mean', # Churn rate per cluster
    'country': lambda x: x.mode()[0], # Most frequent country
    'gender': lambda x: x.mode()[0], # Most frequent gender
    'age_group': lambda x: x.mode()[0], # Most frequent age group
    'balance_to_salary': 'mean',
    'is_zero_balance': 'mean',
    'products_per_year': 'mean'
}).round(2)
    # eg. For each cluster, take the country column and return the most frequent value. Mean but fot categorical variables
# Adding cluster size to the profile
cluster_profile['size'] = df_clustered['cluster'].value_counts().sort_index()

print(cluster_profile.to_markdown(numalign="left", stralign="left"))


# Visualizing Clusters with PCA
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_clustering_processed)

# Reducing dimensionality to 2 components for visualization
pca_df = pd.DataFrame(data=X_pca, columns=['PCA Component 1', 'PCA Component 2'])
pca_df['cluster'] = cluster_labels

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='PCA Component 1',
    y='PCA Component 2',
    hue='cluster',
    palette='viridis',
    data=pca_df,
    legend='full',
    alpha=0.7
)
plt.title(f'Customer Clusters (K={chosen_k}) Visualized with PCA')
plt.xlabel(f'PCA Component 1 (explains {pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PCA Component 2 (explains {pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.grid(True)
plt.show()
